# Exploratory analysis of myocardial infarction complications

This notebook examines patient characteristics, recorded treatments, and
complications in the [UCI Myocardial Infarction Complications dataset](https://archive.ics.uci.edu/dataset/579/myocardial+infarction+complications).
It contains descriptive subgroup comparisons, four-cluster K-means with PCA
visualization, and a random forest baseline for pulmonary edema (`OTEK_LANC`).

Run the cells in order from a fresh Python kernel after installing the
repository requirements. Data loading uses the repository's validated schema
and preserves missing values. Dataset field names are retained so every
calculation can be traced to the source documentation.

These observational comparisons do not estimate treatment effects. No
statistical interaction tests or clinical validation are performed.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Patch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'mi_complications').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mi_complications.data import (
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    feature_columns,
    load_data,
)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300})


## 1. Load data and inspect missingness

The identifier, 111 predictors, and 12 recorded outcomes have distinct roles.
`LET_IS` contains a survival/death-cause code: zero denotes survival and
positive codes denote death. Its numeric mean is not a mortality rate.
We report the 11 binary complications alongside a derived `Mortality`
indicator, retaining missing outcomes as missing.

An earlier Kaggle CSV is not included in the repository and its preprocessing
is undocumented. Its saved outputs are not used as verified results for this
version; running these cells recomputes all results from the declared input.


In [ ]:
data = load_data()
binary_complications = [name for name in TARGET_COLUMNS if name != 'LET_IS']
outcomes = data[binary_complications].copy()
outcomes['Mortality'] = data['LET_IS'].ne(0).astype(float).where(
    data['LET_IS'].notna()
)
outcome_columns = outcomes.columns.tolist()

print(f'Patients: {len(data):,}; recorded columns: {data.shape[1]}')
display(data.head())
missingness = data[list(FEATURE_COLUMNS)].isna().mean().sort_values(
    ascending=False
).rename('Missing fraction')
display(missingness.to_frame().head(15))

admission_data = data[feature_columns('admission')]
display(admission_data.describe())
display(outcomes.agg(['count', 'mean']).T.rename(
    columns={'count': 'Observed outcomes', 'mean': 'Observed proportion'}
).sort_values('Observed proportion', ascending=False))


## 2. Exploratory patient clustering

K-means uses all 111 predictors, mean imputation, standardization, four
clusters, 10 initializations, and seed 42, as in the original analysis. The
full cohort is used because this is descriptive clustering, without a
predictive holdout claim. Later hospital observations are included, so these
clusters are not admission-only profiles.

Numeric encodings include categorical and ordinal variables. Euclidean
distance and mean imputation are exploratory choices, and cluster stability
has not been established. Cluster numbers are arbitrary. The displayed
features have the largest absolute standardized cluster means; they are not
validated clinical labels or causal explanations.


In [ ]:
imputer = SimpleImputer(strategy='mean', keep_empty_features=True)
imputed_features = imputer.fit_transform(data[list(FEATURE_COLUMNS)])
scaler = StandardScaler()
standardized_features = pd.DataFrame(
    scaler.fit_transform(imputed_features),
    columns=FEATURE_COLUMNS,
    index=data.index,
)
kmeans = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
cluster_assignments = kmeans.fit_predict(standardized_features)
cluster_means = standardized_features.assign(
    Cluster=cluster_assignments
).groupby('Cluster').mean()
top_cluster_features = pd.DataFrame({
    f'Cluster {cluster}': means.abs().nlargest(10).index.tolist()
    for cluster, means in cluster_means.iterrows()
})
display(top_cluster_features)
display(cluster_means)


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
principal_components = pca.fit_transform(standardized_features)
explained_variance = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(10, 6))
for cluster, color in enumerate(sns.color_palette('colorblind', 4)):
    selected = cluster_assignments == cluster
    ax.scatter(
        principal_components[selected, 0],
        principal_components[selected, 1],
        label=f'Cluster {cluster} (n={selected.sum()})',
        color=color,
        alpha=0.7,
        s=28,
    )
ax.set(
    title='K-means clusters projected onto two principal components',
    xlabel=f'Principal component 1 ({explained_variance[0]:.1%} variance)',
    ylabel=f'Principal component 2 ({explained_variance[1]:.1%} variance)',
)
ax.legend(title='Cluster')
fig.tight_layout()
plt.show()


## 3. Complications and recorded treatments by subgroup

Proportions use nonmissing observations within each group. Missing age or
hypertension information forms an explicit `Unknown` group in the broad
subgroup summaries; missing sex is handled in the same way. Hypertension
follows the original grouping (`GB > 0`). These summaries are unadjusted.

Treatment fields have different encodings: some are binary and others record
frequency. Their mean recorded codes are displayed for continuity with the
original exploration, and should not be compared as treatment prevalence
across fields or interpreted as a common dose scale.


In [ ]:
analysis_data = pd.concat([data[list(FEATURE_COLUMNS)], outcomes], axis=1)
analysis_data['Age group'] = pd.cut(
    data['AGE'], bins=[-np.inf, 60, np.inf], right=False,
    labels=['Under 60', '60 and above'],
).astype('string').fillna('Unknown')
analysis_data['Sex'] = data['SEX'].map({0: 'Female', 1: 'Male'}).fillna(
    'Unknown'
)
analysis_data['Hypertension'] = data['GB'].gt(0).map(
    {True: 'Yes', False: 'No'}
).where(data['GB'].notna(), 'Unknown')
subgroup_columns = ['Age group', 'Sex', 'Hypertension']
treatment_columns = [
    'NA_KB', 'NOT_NA_KB', 'LID_KB', 'NITR_S',
    'NA_R_1_n', 'NA_R_2_n', 'NA_R_3_n',
    'NOT_NA_1_n', 'NOT_NA_2_n', 'NOT_NA_3_n',
    'LID_S_n', 'B_BLOK_S_n', 'ANT_CA_S_n', 'GEPAR_S_n',
    'ASP_S_n', 'TIKL_S_n', 'TRENT_S_n',
]


def plot_subgroup_means(frame, columns, title, colorbar_label, vmax=None):
    """Plot observed means using a common color scale across subgroups."""
    summaries = {
        group: frame.groupby(group, observed=True)[columns].mean()
        for group in subgroup_columns
    }
    if vmax is None:
        vmax = max(summary.max().max() for summary in summaries.values())
    fig, axes = plt.subplots(1, 3, figsize=(17, 8))
    for ax, (group, summary) in zip(axes, summaries.items()):
        sns.heatmap(
            summary.T, annot=True, fmt='.2f', cmap='YlGnBu',
            vmin=0, vmax=vmax, ax=ax,
            cbar_kws={'label': colorbar_label},
        )
        counts = frame.groupby(group, observed=True).size()
        ax.set_xticklabels([
            f'{label} (n={counts[label]})' for label in summary.index
        ], rotation=30, ha='right')
        ax.set(title=f'By {group.lower()}', xlabel=group, ylabel='')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return summaries


complication_subgroups = plot_subgroup_means(
    analysis_data, outcome_columns, 'Observed complication proportions',
    'Proportion', vmax=1,
)
treatment_subgroups = plot_subgroup_means(
    analysis_data, treatment_columns, 'Mean recorded treatment codes',
    'Mean code',
)


## 4. Unadjusted treatment-code comparisons

The individual comparisons retain the original code-1 versus code-0
definition. Higher frequency codes and missing exposure records are excluded
and counted, rather than assigned to the untreated group. For a
frequency-coded field, code 1 is not equivalent to all treatment exposure.

The summary difference is the mean across the 11 binary complications and
the mortality indicator, calculated as **code 0 minus code 1**. This is an
exploratory summary across different outcomes, not a validated clinical
endpoint. Positive values describe a lower observed mean in the code-1
group. Confounding by indication, treatment timing, missingness, and case mix
prevent a treatment-effect interpretation.


In [ ]:
comparison_rows = []
treatment_outcome_comparisons = {}
for treatment in treatment_columns:
    code_one = data[treatment].eq(1)
    code_zero = data[treatment].eq(0)
    mean_one = outcomes.loc[code_one].mean()
    mean_zero = outcomes.loc[code_zero].mean()
    differences = mean_zero - mean_one
    treatment_outcome_comparisons[treatment] = pd.DataFrame({
        'Code 1 proportion': mean_one,
        'Code 0 proportion': mean_zero,
        'Code 0 minus code 1': differences,
    })
    comparison_rows.append({
        'Treatment': treatment,
        'Code 1 patients': int(code_one.sum()),
        'Code 0 patients': int(code_zero.sum()),
        'Excluded patients': int((~(code_one | code_zero)).sum()),
        'Mean difference': differences.mean(),
    })
treatment_comparisons = pd.DataFrame(comparison_rows).set_index('Treatment')
treatment_comparisons = treatment_comparisons.sort_values(
    'Mean difference', ascending=False
)
display(treatment_comparisons)

selected_comparisons = pd.concat([
    treatment_comparisons.head(5), treatment_comparisons.tail(5)
])
values = selected_comparisons['Mean difference']
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(values.index, values, color=np.where(values >= 0, '#0072B2', '#D55E00'))
ax.axhline(0, color='black', linewidth=0.8)
ax.set(
    title='Largest and smallest unadjusted mean differences',
    xlabel='Recorded treatment field',
    ylabel='Mean proportion difference (code 0 minus code 1)',
)
ax.tick_params(axis='x', rotation=45)
plt.setp(ax.get_xticklabels(), ha='right')
ax.legend(handles=[
    Patch(color='#0072B2', label='Lower observed mean in code 1'),
    Patch(color='#D55E00', label='Higher observed mean in code 1'),
])
fig.tight_layout()
plt.show()


## 5. Pulmonary edema prediction

This baseline preserves the original 11 predictors, complete-case selection,
a 70/30 random split with seed 42, and a default random forest with seed 42.
The duplicate fit has been removed. Missing target values are also excluded.
The split remains unstratified to retain the original design; sample sizes
and class proportions are shown to make its limitations visible.

Accuracy and ROC-AUC describe this single holdout. There is no tuning,
cross-validation, calibration assessment, or external validation. Complete-case
selection can change the studied population, and predictor timing has not
been validated for a specific clinical prediction time. Impurity-based
feature importance is a model diagnostic, not evidence of causation.


In [ ]:
prediction_features = [
    'AGE', 'SEX', 'GB', 'INF_ANAM', 'STENOK_AN', 'FK_STENOK',
    'S_AD_KBRIG', 'D_AD_KBRIG', 'L_BLOOD', 'K_BLOOD', 'DLIT_AG',
]
target_column = 'OTEK_LANC'
model_data = data[prediction_features + [target_column]].dropna()
X_train, X_test, y_train, y_test = train_test_split(
    model_data[prediction_features], model_data[target_column],
    test_size=0.3, random_state=RANDOM_STATE,
)
print(f'Complete cases: {len(model_data):,} of {len(data):,} patients')
print(f'Training patients: {len(X_train):,}; test patients: {len(X_test):,}')
print(f'Training prevalence: {y_train.mean():.3f}; test prevalence: {y_test.mean():.3f}')

model = RandomForestClassifier(random_state=RANDOM_STATE)
model.fit(X_train, y_train)
predicted_labels = model.predict(X_test)
if 1 in model.classes_:
    positive_class = list(model.classes_).index(1)
    predicted_probabilities = model.predict_proba(X_test)[:, positive_class]
else:
    predicted_probabilities = np.zeros(len(X_test))
test_roc_auc = (
    roc_auc_score(y_test, predicted_probabilities)
    if y_test.nunique() == 2 else np.nan
)
model_metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, predicted_labels),
    'ROC-AUC': test_roc_auc,
}, name='Single holdout')
display(model_metrics.to_frame())
if y_test.nunique() != 2:
    print('ROC-AUC is undefined because the test set contains only one class.')

feature_importance = pd.Series(
    model.feature_importances_, index=prediction_features,
    name='Impurity-based importance',
).sort_values(ascending=False)
display(feature_importance.to_frame())

fig, ax = plt.subplots(figsize=(9, 6))
feature_importance.sort_values().plot.barh(ax=ax, color='#0072B2')
ax.set(
    title='Random forest feature importance for pulmonary edema',
    xlabel='Impurity-based importance', ylabel='Predictor',
)
fig.tight_layout()
plt.show()


## 6. Recorded treatment combinations

These three combinations use binary treatment fields. Only patients with
observed code 0 or 1 for both fields are included. Both code 1 is compared
with all other observed combinations; the original input columns are never
converted in place to Boolean values. In particular, missing treatment
information is not treated as exposure.

Here the difference follows the original combination analysis: **both code 1
minus other combinations**. It has the opposite direction to the individual
comparison in Section 4. Differences are descriptive associations.


In [ ]:
combination_pairs = {
    'B_BLOK_S_n + ASP_S_n': ('B_BLOK_S_n', 'ASP_S_n'),
    'B_BLOK_S_n + ANT_CA_S_n': ('B_BLOK_S_n', 'ANT_CA_S_n'),
    'ASP_S_n + ANT_CA_S_n': ('ASP_S_n', 'ANT_CA_S_n'),
}
combination_summaries = {}
combination_counts = []
for label, pair in combination_pairs.items():
    known = data[list(pair)].isin([0, 1]).all(axis=1)
    both = data[list(pair)].eq(1).all(axis=1)
    both_mean = outcomes.loc[known & both].mean()
    other_mean = outcomes.loc[known & ~both].mean()
    combination_summaries[label] = pd.DataFrame({
        'Both code 1': both_mean,
        'Other observed combinations': other_mean,
        'Both minus other': both_mean - other_mean,
    })
    combination_counts.append({
        'Combination': label,
        'Both code 1': int((known & both).sum()),
        'Other combinations': int((known & ~both).sum()),
        'Excluded': int((~known).sum()),
    })
display(pd.DataFrame(combination_counts).set_index('Combination'))

fig, axes = plt.subplots(3, 1, figsize=(15, 14))
for ax, measure in zip(axes, next(iter(combination_summaries.values())).columns):
    summary = pd.DataFrame({
        label: table[measure] for label, table in combination_summaries.items()
    }).T
    summary.plot.bar(ax=ax, colormap='tab20', width=0.85)
    ax.set(
        title=measure, xlabel='',
        ylabel='Proportion difference' if measure == 'Both minus other' else 'Proportion',
    )
    ax.tick_params(axis='x', rotation=0)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
plt.show()


## 7. Selected complications by age group and sex

The original six complications and age bins are retained. Records missing
age or sex are excluded from this table. Group counts accompany the observed
proportions; small subgroups and unadjusted comparisons require caution.
The repeated styling experiments have been consolidated into one grouped
bar chart per complication.


In [ ]:
demographic_complications = [
    'FIBR_PREDS', 'PREDS_TAH', 'JELUD_TAH',
    'FIBR_JELUD', 'A_V_BLOK', 'OTEK_LANC',
]
age_labels = ['Under 50', '50-59', '60-69', '70-79', '80+']
demographic_data = analysis_data.copy()
demographic_data['Age band'] = pd.cut(
    data['AGE'], bins=[0, 50, 60, 70, 80, 100],
    labels=age_labels, right=False,
)
demographic_data = demographic_data.loc[
    demographic_data['Age band'].notna() & data['SEX'].isin([0, 1])
]
grouped_demographics = demographic_data.groupby(
    ['Age band', 'Sex'], observed=True
)
demographic_proportions = grouped_demographics[demographic_complications].mean()
demographic_counts = grouped_demographics.size().rename('Patients')
display(demographic_proportions.join(demographic_counts))
print(f'Excluded demographic records: {len(data) - len(demographic_data)}')

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for complication, ax in zip(demographic_complications, axes.flat):
    plot_data = demographic_proportions[complication].unstack('Sex').reindex(
        index=age_labels, columns=['Female', 'Male']
    )
    plot_data.plot.bar(ax=ax, color=['#0072B2', '#D55E00'], width=0.8)
    ax.set(title=complication, xlabel='Age group', ylabel='Observed proportion')
    ax.tick_params(axis='x', rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=2, fontsize=8)
    ax.margins(y=0.15)
    ax.legend(title='Sex')
fig.suptitle('Selected complications by age group and sex')
fig.tight_layout()
plt.show()


## 8. Mortality by pairs of recorded treatment codes

These summaries retain the three treatment pairs from the original notebook.
They report the observed survival/death proportions, not survival curves,
treatment effects, or statistically significant interactions. Death includes
every positive `LET_IS` code. Patients with missing outcomes, missing
exposure codes, or exposure codes outside 0 and 1 are excluded for each pair.
This prevents higher frequency codes from being mislabeled as no treatment.


In [ ]:
def summarize_treatment_pair(frame, first, second):
    """Summarize mortality for observed code-0/code-1 treatment pairs."""
    valid = frame[[first, second]].isin([0, 1]).all(axis=1)
    valid &= frame['LET_IS'].notna()
    pair_data = frame.loc[valid, [first, second, 'LET_IS']].copy()
    pair_data['Mortality'] = pair_data['LET_IS'].gt(0).astype(float)
    grouped = pair_data.groupby([first, second])['Mortality'].agg(['size', 'mean'])
    index = pd.MultiIndex.from_tuples(
        [(1, 1), (1, 0), (0, 1), (0, 0)], names=[first, second]
    )
    summary = grouped.reindex(index).rename(
        columns={'size': 'Patients', 'mean': 'Mortality'}
    )
    summary['Patients'] = summary['Patients'].fillna(0).astype(int)
    summary['Survival'] = 1 - summary['Mortality']
    return summary, int((~valid).sum())


interaction_pairs = [
    ('NA_R_1_n', 'NITR_S'),
    ('NOT_NA_1_n', 'ANT_CA_S_n'),
    ('ASP_S_n', 'GEPAR_S_n'),
]
pair_summaries = {}
for first, second in interaction_pairs:
    summary, excluded = summarize_treatment_pair(data, first, second)
    pair_summaries[(first, second)] = summary
    print(f'{first} and {second}: {excluded} excluded records')
    display(summary)
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_data = summary[['Survival', 'Mortality']].copy()
    plot_data.index = [
        f'{first_code}, {second_code} (n={count})'
        for (first_code, second_code), count in zip(summary.index, summary['Patients'])
    ]
    plot_data.plot.bar(ax=ax, color=['#0072B2', '#D55E00'], width=0.8)
    ax.set(
        title=f'Observed outcomes: {first} and {second}',
        xlabel=f'Recorded codes ({first}, {second})',
        ylabel='Observed proportion', ylim=(0, 1.05),
    )
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Outcome')
    fig.tight_layout()
    plt.show()


## Interpretation and provenance notes

The original placeholder plots averaged unrelated tabular features across
patients and labeled them ECG traces, including arbitrary P/Q/R/S/T
annotations. They have been removed because these calculations cannot
represent physiological waveforms. PCA axes now describe components rather
than individual clinical measurements. Admission feature inspection excludes
identifiers, targets, and later observations specified by the source schema.

Historical outputs were cleared because input provenance and corrected
outcome definitions make the old figures and metrics unsuitable for the
current code. The earlier notebook remains available in Git history.

The original manually entered recommendation chart was not derived from a
model or a clinical evidence review. Its notes are retained below solely as
an unvalidated record of that draft. They are not findings or guidance from
this analysis, and the arbitrary numeric bar heights have been removed.

| Age group | Original topic | Unvalidated draft note |
| --- | --- | --- |
| Under 50 | Atrial fibrillation | Education and monitoring |
| Under 50 | Pulmonary edema | Frequent monitoring |
| 50-59 | Atrial fibrillation | Antiarrhythmic medication and beta blockers |
| 50-59 | Pulmonary edema | Monitoring patients with hypertension |
| 60-69 | Ventricular tachycardia | ECG monitoring |
| 60-69 | Pulmonary edema | Risk-factor management |
| 70-79 | Atrial fibrillation | Anticoagulant follow-up |
| 70-79 | Atrioventricular block | Holter ECG monitoring |
| 80+ | Atrial fibrillation | Individualized treatment |
| 80+ | Pulmonary edema | Intensive home monitoring |

Further work would require validated feature timing, sensitivity analyses for
missingness and treatment encodings, uncertainty estimates, and independent
model validation. These steps have not been performed here.
